In [152]:
import math

import pandas as pd
from tabulate import tabulate


## Question 1

```
a	b	c	p(a, b, c)
0	0	0	0.192
0	0	1	0.144
0	1	0	0.048
0	1	1	0.216
1	0	0	0.192
1	0	1	0.064
1	1	0	0.048
1	1	1	0.096
```

Are a and b independent? (Test whether p(a ∩ b) = p(a)·p(b).)

In [153]:
prob_table = [
    [0, 0, 0, 0.192],
    [0, 0, 1, 0.144],
    [0, 1, 0, 0.048],
    [0, 1, 1, 0.216],
    [1, 0, 0, 0.192],
    [1, 0, 1, 0.064],
    [1, 1, 0, 0.048],
    [1, 1, 1, 0.096],
]

joint_prob_df = pd.DataFrame(prob_table, columns=["a", "b", "c", "prob"])

print(
    tabulate(
        joint_prob_df,
        headers="keys",
        floatfmt=[".0f", ".0f", ".0f", ".3f"],
        showindex=False,
    )
)

  a    b    c    prob
---  ---  ---  ------
  0    0    0   0.192
  0    0    1   0.144
  0    1    0   0.048
  0    1    1   0.216
  1    0    0   0.192
  1    0    1   0.064
  1    1    0   0.048
  1    1    1   0.096


The value of $p(a, b) = p(a, b, c) + p(a, b, \neg c)$.

In [154]:
joint_ab = joint_prob_df.groupby(by=["a", "b"])["prob"].sum().reset_index()

print(
    tabulate(
        joint_ab,
        headers=["a", "b", "prob"],
        floatfmt=[".0f", ".0f", ".3f"],
        showindex=False,
    )
)

  a    b    prob
---  ---  ------
  0    0   0.336
  0    1   0.264
  1    0   0.256
  1    1   0.144


The value of $p(a) = p(a, b) + p(a, \neg b)$ and $p(b) = p(a, b) + p(\neg a, b)$.

In [155]:
marginal_a = joint_prob_df.groupby(by=["a"])["prob"].sum().reset_index()

print(
    tabulate(
        marginal_a, headers=["a", "prob"], floatfmt=[".0f", ".3f"], showindex=False
    )
)

  a    prob
---  ------
  0   0.600
  1   0.400


In [156]:
marginal_b = joint_prob_df.groupby(by=["b"])["prob"].sum().reset_index()

print(
    tabulate(
        marginal_b, headers=["b", "prob"], floatfmt=[".0f", ".3f"], showindex=False
    )
)

  b    prob
---  ------
  0   0.592
  1   0.408


If $p(a, b) = p(a)p(b)$, then the entries in the previously computed $p(a, b)$ from marginalization should be the same as multiplying the singular marginals.

In [157]:
question_1_values = []

for a in joint_prob_df.a.unique():
    for b in joint_prob_df.b.unique():
        prob_joint = joint_ab.query(f"a == {a} and b == {b}").prob.values[0]
        prob_product = (
            marginal_a.query(f"a == {a}").prob.values[0]
            * marginal_b.query(f"b == {b}").prob.values[0]
        )
        are_equal = math.isclose(prob_joint, prob_product)
        question_1_values.append([a, b, prob_joint, prob_product, are_equal])

print(
    tabulate(
        question_1_values,
        headers=["a", "b", "p(a, b)", "p(a) * p(b)", "equal?"],
        floatfmt=".3f",
    )
)

  a    b    p(a, b)    p(a) * p(b)  equal?
---  ---  ---------  -------------  --------
  0    0      0.336          0.355  False
  0    1      0.264          0.245  False
  1    0      0.256          0.237  False
  1    1      0.144          0.163  False


**The value of $p(a, b)$ is *not* independent since $p(a, b) \ne p(a) p(b)$.**

## Question 2

```
a	b	c	p(a, b, c)
0	0	0	0.192
0	0	1	0.144
0	1	0	0.048
0	1	1	0.216
1	0	0	0.192
1	0	1	0.064
1	1	0	0.048
1	1	1	0.096
```

Are a and b conditionally independent given c? (Test whether p(a ∩ b | c) = p(a | c)·p(b | c).)

By the formula for conditional probability, $p(a, b \mid c) = \frac{p(a, b, c)}{p(c)}$. For the denominator, marginalizing the joint should provide $p(c)$:

$$
\begin{align*}
    p(c) &= \sum_{a}\sum_{b} p(a, b, c)
\end{align*}
$$

In [158]:
marginal_c = joint_prob_df.groupby(by=["c"]).prob.sum().reset_index()

print(
    tabulate(
        marginal_c, headers=["c", "prob"], floatfmt=[".0f", ".3f"], showindex=False
    )
)

  c    prob
---  ------
  0   0.480
  1   0.520


Using this to find $p(a, b \mid c)$ and $p(a, b \mid \neg c)$:

In [159]:
joint_ab_given_c = (
    joint_prob_df.query("c == 1").groupby(by=["a", "b"]).prob.sum().reset_index()
)
joint_ab_given_c.prob = (
    joint_ab_given_c.prob / marginal_c.query("c == 1").prob.values[0]
)

print("p(a, b | c = 0):")
print(
    tabulate(
        joint_ab_given_c,
        headers=["a", "b", "prob"],
        floatfmt=[".0f", ".0f", ".3f"],
        showindex=False,
    )
)

p(a, b | c = 0):
  a    b    prob
---  ---  ------
  0    0   0.277
  0    1   0.415
  1    0   0.123
  1    1   0.185


In [160]:
joint_ab_given_not_c = (
    joint_prob_df.query("c == 0").groupby(by=["a", "b"]).prob.sum().reset_index()
)
joint_ab_given_not_c.prob = (
    joint_ab_given_not_c.prob / marginal_c.query("c == 0").prob.values[0]
)

print("p(a, b | c = 0):")
print(
    tabulate(
        joint_ab_given_not_c,
        headers=["a", "b", "prob"],
        floatfmt=[".0f", ".0f", ".3f"],
        showindex=False,
    )
)

p(a, b | c = 0):
  a    b    prob
---  ---  ------
  0    0   0.400
  0    1   0.100
  1    0   0.400
  1    1   0.100


Finding the conditionals for $p(a | c)$ and $p(b | c)$ is the same marginalization exercise as before, but with the conditionals:

$$
\begin{align*}
    p(a \mid c) &= \sum_{b} p(a, b \mid  c) \\
    p(b \mid c) &= \sum_{a} p(a, b \mid  c) \\
\end{align*}
$$


In [161]:
marginal_a_given_c = joint_ab_given_c.groupby(by=["a"]).prob.sum().reset_index()
print("p(a | c = 1):")
print(
    tabulate(
        marginal_a_given_c,
        headers=["a", "prob"],
        floatfmt=[".0f", ".3f"],
        showindex=False,
    )
)

p(a | c = 1):
  a    prob
---  ------
  0   0.692
  1   0.308


In [162]:
marginal_a_given_not_c = joint_ab_given_not_c.groupby(by=["a"]).prob.sum().reset_index()
print("p(a | c = 0):")
print(
    tabulate(
        marginal_a_given_not_c,
        headers=["a", "prob"],
        floatfmt=[".0f", ".3f"],
        showindex=False,
    )
)

p(a | c = 0):
  a    prob
---  ------
  0   0.500
  1   0.500


In [163]:
marginal_b_given_c = joint_ab_given_c.groupby(by=["b"]).prob.sum().reset_index()
print("p(b | c = 1):")
print(
    tabulate(
        marginal_b_given_c,
        headers=["b", "prob"],
        floatfmt=[".0f", ".3f"],
        showindex=False,
    )
)

p(b | c = 1):
  b    prob
---  ------
  0   0.400
  1   0.600


In [164]:
marginal_b_given_not_c = joint_ab_given_not_c.groupby(by=["b"]).prob.sum().reset_index()
print("p(b | c = 0):")
print(
    tabulate(
        marginal_b_given_not_c,
        headers=["b", "prob"],
        floatfmt=[".0f", ".3f"],
        showindex=False,
    )
)

p(b | c = 0):
  b    prob
---  ------
  0   0.800
  1   0.200


Now checking if $p(a, b \mid c) = p(a \mid c) p(b \mid c)$:

In [165]:
question_2_values_c = []

for a in joint_prob_df.a.unique():
    for b in joint_prob_df.b.unique():
        prob_conditional_joint = joint_ab_given_c.query(
            f"a == {a} and b == {b}"
        ).prob.values[0]
        prob_conditional_product = (
            marginal_a_given_c.query(f"a == {a}").prob.values[0]
            * marginal_b_given_c.query(f"b == {b}").prob.values[0]
        )
        are_equal = math.isclose(prob_conditional_joint, prob_conditional_product)
        question_2_values_c.append(
            [a, b, prob_conditional_joint, prob_conditional_product, are_equal]
        )

print(
    tabulate(
        question_2_values_c,
        headers=["a", "b", "p(a, b | c = 1)", "p(a | c = 1) * p(b | c = 1)", "equal?"],
        floatfmt=".3f",
    )
)

  a    b    p(a, b | c = 1)    p(a | c = 1) * p(b | c = 1)  equal?
---  ---  -----------------  -----------------------------  --------
  0    0              0.277                          0.277  True
  0    1              0.415                          0.415  True
  1    0              0.123                          0.123  True
  1    1              0.185                          0.185  True


In [166]:
question_2_values_not_c = []

for a in joint_prob_df.a.unique():
    for b in joint_prob_df.b.unique():
        prob_conditional_joint = joint_ab_given_not_c.query(
            f"a == {a} and b == {b}"
        ).prob.values[0]
        prob_conditional_product = (
            marginal_a_given_not_c.query(f"a == {a}").prob.values[0]
            * marginal_b_given_not_c.query(f"b == {b}").prob.values[0]
        )
        are_equal = math.isclose(prob_conditional_joint, prob_conditional_product)
        question_2_values_not_c.append(
            [a, b, prob_conditional_joint, prob_conditional_product, are_equal]
        )

print(
    tabulate(
        question_2_values_not_c,
        headers=["a", "b", "p(a, b | c = 0)", "p(a | c = 0) * p(b | c = 0)", "equal?"],
        floatfmt=".3f",
    )
)

  a    b    p(a, b | c = 0)    p(a | c = 0) * p(b | c = 0)  equal?
---  ---  -----------------  -----------------------------  --------
  0    0              0.400                          0.400  True
  0    1              0.100                          0.100  True
  1    0              0.400                          0.400  True
  1    1              0.100                          0.100  True


**In this case, $p(a, b \mid c) = p(a \mid c) p(b \mid c)$ is true**.

## Question 3

